# Pandas Revision

**Covers:**
1. Series — 1D labelled array
2. DataFrame — 2D labelled table
3. Indexing with `loc` and `iloc`
   - Single value
   - Two values (slicing)
   - Fancy indexing (lists, boolean masks)
4. `groupby`
5. `map`, `apply`, `applymap`, `transform`

Optimised for quick revision — concise code + key concepts.

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 10)

In [3]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9]
})
df

,A,B,C
0,1,4,7
1,2,5,8
2,3,6,9


In [5]:
type(df.sum())

pandas.core.series.Series

## 1 — Series

A **Series** is a 1D labelled array. Think of it as a column with an index.

```
Series = index + values
```

In [6]:
# Create from a list (default integer index)
s = pd.Series([10, 20, 30, 40, 50])
print(s)

0    10
1    20
2    30
3    40
4    50
dtype: int64


In [3]:
# Create with custom index
s = pd.Series([10, 20, 30, 40, 50], index=['a', 'b', 'c', 'd', 'e'])
print(s)

a    10
b    20
c    30
d    40
e    50
dtype: int64


In [4]:
# Access by label or position
print(s['a'])     # label
print(s[0])       # position (deprecated for label-based but still works)
print(s.values)   # numpy array of values
print(s.index)    # the index

10
10
[10 20 30 40 50]
Index(['a', 'b', 'c', 'd', 'e'], dtype='object')


/tmp/ipykernel_532251/3759578855.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(s[0])       # position (deprecated for label-based but still works)


## 2 — DataFrame

A **DataFrame** is a 2D labelled table. Think of multiple Series sharing the same index.

```
DataFrame = rows (index) × columns (column names) × values
```

In [60]:
# Create from dict — each key becomes a column
data = {
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':    [25,     30,    35,      40,    45],
    'salary': [50000, 60000, 75000, 90000, 100000],
    'city':   ['NYC', 'LA', 'NYC', 'SF', 'LA']
}
df = pd.DataFrame(data)
df

,name,age,salary,city
0,Alice,25,50000,NYC
1,Bob,30,60000,LA
2,Carol,35,75000,NYC
3,Dave,40,90000,SF
4,Eve,45,100000,LA


In [6]:
# Set a custom index
df.index = ['e1', 'e2', 'e3', 'e4', 'e5']
df

,name,age,salary,city
e1,Alice,25,50000,NYC
e2,Bob,30,60000,LA
e3,Carol,35,75000,NYC
e4,Dave,40,90000,SF
e5,Eve,45,100000,LA


### Adding Rows to a DataFrame

Several ways to append rows. Use `pd.concat` for the modern, recommended approach.

| Method | Use case |
|--------|----------|
| `pd.concat([df, new_row])` | **Recommended** — adding 1+ rows |
| `df.loc[new_label] = [...]` | Quick single-row append (in place) |
| `df.loc[new_label] = {...}` | Same but with a dict (clearer) |
| `df.append(...)` | ⚠️ Removed in pandas 2.0 — don't use |

In [40]:
# Method 1: pd.concat — recommended, works for 1 or many rows
new_row = pd.DataFrame(
    [{'name': 'Frank', 'age': 28, 'salary': 55000, 'city': 'NYC'}],
    index=['e6']
)
df2 = pd.concat([df, new_row])
df2

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF
e5,Carol,35,NaN,None
e6,Frank,28,55000.0,NYC


In [41]:
# Method 2: df.loc[new_label] — list of values (in place)
df2.loc['e7'] = ['Grace', 32, 70000, 'LA']
df2

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF
e5,Carol,35,NaN,None
e6,Frank,28,55000.0,NYC
e7,Grace,32,70000.0,LA


In [42]:
# Method 3: df.loc[new_label] = {dict} — clearer, column order doesn't matter
df2.loc['e8'] = {'name': 'Henry', 'salary': 80000, 'city': 'SF', 'age': 38}
df2

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF
e5,Carol,35,NaN,None
e6,Frank,28,55000.0,NYC
e7,Grace,32,70000.0,LA
e8,Henry,38,80000.0,SF


In [48]:
new_row = pd.DataFrame(
    [{'name': 'Liam', 'age': 28, 'salary': 34000, 'city': 'NYC'}]
)
df3 = pd.concat([df2, new_row], ignore_index=True )
df3

,name,age,salary,city
0,Alice,25,50000.0,NYC
1,Bob,30,60000.0,LA
2,Carol,35,75000.0,NYC
3,Dave,40,90000.0,SF
4,Carol,35,NaN,None
5,Frank,28,55000.0,NYC
6,Grace,32,70000.0,LA
7,Henry,38,80000.0,SF
8,Liam,28,34000.0,NYC


**Performance tip — adding many rows:**

```python
# ❌ SLOW — concat in a loop (re-allocates each time, O(n²))
for record in records:
    df = pd.concat([df, pd.DataFrame([record])], ignore_index=True)

# ✅ FAST — collect records first, concat ONCE
records = [{'name': 'A', 'age': 20, 'salary': 30000, 'city': 'NYC'},
           {'name': 'B', 'age': 25, 'salary': 40000, 'city': 'LA'}]
df = pd.concat([df, pd.DataFrame(records)], ignore_index=True)
```

## 3 — loc and iloc

Two ways to index pandas:

```
loc   → LABEL-based  (uses index names, column names)
iloc  → POSITION-based (uses integer position 0, 1, 2, ...)
```

Both support:
- Single value
- Two values (row + column)
- Slices (a:b)
- Fancy indexing (lists, boolean masks)

### 3a — Single value (one row, all columns)

In [7]:
# loc — by label
print(df.loc['e1'])
print('---')
# iloc — by position
print(df.iloc[0])

name      Alice
age          25
salary    50000
city        NYC
Name: e1, dtype: object
---
name      Alice
age          25
salary    50000
city        NYC
Name: e1, dtype: object


In [ ]:
try:
    df['e1']
    df[0]
except KeyError:
    print("KeyError")

KeyError


In [ ]:
# In pandas, df[...] is overloaded:

# 1. Single brackets: df['col']
# returns a Series
# selects one column
# 2. Double brackets: df[['col1', 'col2']]
# returns a DataFrame
# selects multiple columns

In [24]:
df["salary"]

e1     50000
e2     60000
e3     75000
e4     90000
e5    100000
Name: salary, dtype: int64

In [25]:
df[["name", "salary"]]

,name,salary
e1,Alice,50000
e2,Bob,60000
e3,Carol,75000
e4,Dave,90000
e5,Eve,100000


### 3b — Two values (specific row + column)

In [8]:
# loc — labels: row label, column label
print(df.loc['e2', 'salary'])     # Bob's salary

# iloc — positions: row index, column index
print(df.iloc[1, 2])              # row 1, column 2 — Bob's salary

60000
60000


### 3c — Slicing (range of rows / columns)

In [9]:
# loc slicing INCLUDES the endpoint
df.loc['e1':'e3']    # rows e1, e2, e3

,name,age,salary,city
e1,Alice,25,50000,NYC
e2,Bob,30,60000,LA
e3,Carol,35,75000,NYC


In [10]:
# iloc slicing EXCLUDES the endpoint (Python convention)
df.iloc[0:3]         # rows 0, 1, 2 — does NOT include row 3

,name,age,salary,city
e1,Alice,25,50000,NYC
e2,Bob,30,60000,LA
e3,Carol,35,75000,NYC


In [11]:
# Slicing rows AND columns
df.loc['e1':'e3', 'name':'salary']    # 3 rows × 3 cols (name, age, salary)

,name,age,salary
e1,Alice,25,50000
e2,Bob,30,60000
e3,Carol,35,75000


In [12]:
# iloc version — both excluded
df.iloc[0:3, 0:3]                    # same 3 rows, first 3 cols

,name,age,salary
e1,Alice,25,50000
e2,Bob,30,60000
e3,Carol,35,75000


### 3d — Fancy indexing (lists)

Pass a **list** of labels or positions to pick specific rows/columns.

In [13]:
# loc — list of labels
df.loc[['e1', 'e3', 'e5']]            # 3 specific rows

,name,age,salary,city
e1,Alice,25,50000,NYC
e3,Carol,35,75000,NYC
e5,Eve,45,100000,LA


In [14]:
# loc — rows AND columns by list
df.loc[['e1', 'e3'], ['name', 'salary']]

,name,salary
e1,Alice,50000
e3,Carol,75000


In [15]:
# iloc — list of positions
df.iloc[[0, 2, 4], [0, 2]]            # rows 0,2,4 — cols 0,2 (name, salary)

,name,salary
e1,Alice,50000
e3,Carol,75000
e5,Eve,100000


In [17]:
df[['name', 'salary']]

,name,salary
e1,Alice,50000
e2,Bob,60000
e3,Carol,75000
e4,Dave,90000
e5,Eve,100000


### 3e — Boolean mask (filter rows by condition)

In [26]:
# Filter rows where salary > 60000
df[df['salary'] > 60000]

,name,age,salary,city
e3,Carol,35,75000,NYC
e4,Dave,40,90000,SF
e5,Eve,45,100000,LA


In [27]:
# Equivalent with loc — also picks columns
df.loc[df['salary'] > 60000, ['name', 'salary']]

,name,salary
e3,Carol,75000
e4,Dave,90000
e5,Eve,100000


In [28]:
df.loc[df['salary'] > 60000, :]

,name,age,salary,city
e3,Carol,35,75000,NYC
e4,Dave,40,90000,SF
e5,Eve,45,100000,LA


In [29]:
# Combine conditions — use & (and), | (or), ~ (not). Always parenthesise!
df[(df['salary'] > 60000) & (df['city'] == 'NYC')]

,name,age,salary,city
e3,Carol,35,75000,NYC


### 3f — All columns, specific rows (using `:`)

In [ ]:
# All columns for specific rows
df.loc['e1':'e3', :]    # same as df.loc['e1':'e3'] #Note e3 is included


,name,age,salary,city
e1,Alice,25,50000,NYC
e2,Bob,30,60000,LA
e3,Carol,35,75000,NYC


In [33]:
df.iloc[:, 1]            # all rows, only column at position 1 (age)

e1    25
e2    30
e3    35
e4    40
e5    45
Name: age, dtype: int64

In [30]:
df.loc['e1']

name      Alice
age          25
salary    50000
city        NYC
Name: e1, dtype: object

### loc vs iloc — Summary

| | loc | iloc |
|-|-----|------|
| Uses | Labels (index name, column name) | Integer positions |
| Slice endpoint | INCLUDED | EXCLUDED |
| Boolean masks | Works | Works |
| Lists | Works | Works |
| When to use | Have meaningful labels | Want positional access |

## 4 — groupby

**Groupby splits data into groups, applies a function, then combines results.**

```
split → apply → combine
```

Powerful for aggregations like "average salary per city".

In [36]:
df

,name,age,salary,city
e1,Alice,25,50000,NYC
e2,Bob,30,60000,LA
e3,Carol,35,75000,NYC
e4,Dave,40,90000,SF
e5,Eve,45,100000,LA


In [52]:
df.loc["e5"] = ['Carol', 35, None, None]
df

,name,age,salary,city
e1,Alice,25,50000.0,NYC
e2,Bob,30,60000.0,LA
e3,Carol,35,75000.0,NYC
e4,Dave,40,90000.0,SF
e5,Carol,35,NaN,None


In [49]:
# Average salary per city
df.groupby('city')['salary'].mean()

city
LA     60000.0
NYC    62500.0
SF     90000.0
Name: salary, dtype: float64

In [53]:
# Multiple aggregations at once
df.groupby('city')['salary'].agg(['mean', 'min', 'max', 'count'])

,mean,min,max,count
city,,,,
LA,60000.0,60000.0,60000.0,1
NYC,62500.0,50000.0,75000.0,2
SF,90000.0,90000.0,90000.0,1


In [57]:
# Aggregate multiple columns with different functions
df.groupby('city').agg({
    # 'salary': lamba x:x+7, = won't work as x is series of column
    'salary': lambda x: x.mean() + 10,
    'age':    'max'
})

,salary,age
city,,
LA,60010.0,30
NYC,62510.0,35
SF,90010.0,40


In [61]:
data = {
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':    [25,     30,    35,      40,    45],
    'salary': [50000, 60000, 75000, 90000, 100000],
    'city':   ['NYC', 'LA', 'NYC', 'SF', 'LA']
}
df = pd.DataFrame(data)
df

,name,age,salary,city
0,Alice,25,50000,NYC
1,Bob,30,60000,LA
2,Carol,35,75000,NYC
3,Dave,40,90000,SF
4,Eve,45,100000,LA


In [65]:
df['dept'] = ['eng', 'sales', 'eng', 'sales', 'eng']
df

,name,age,salary,city,dept
0,Alice,25,50000,NYC,eng
1,Bob,30,60000,LA,sales
2,Carol,35,75000,NYC,eng
3,Dave,40,90000,SF,sales
4,Eve,45,100000,LA,eng


In [70]:
# Group by multiple columns
grouped = df.groupby(['city', 'dept'])['salary'].mean()
grouped


city  dept 
LA    eng      100000.0
      sales     60000.0
NYC   eng       62500.0
SF    sales     90000.0
Name: salary, dtype: float64

In [ ]:
type(grouped)

In [71]:
grouped.loc['LA']

dept
eng      100000.0
sales     60000.0
Name: salary, dtype: float64

In [74]:
grouped.loc['LA', 'eng']

np.float64(100000.0)

In [76]:
grouped['LA']

dept
eng      100000.0
sales     60000.0
Name: salary, dtype: float64

In [73]:
grouped.iloc[0]

np.float64(100000.0)

In [78]:
grouped = df.groupby(['city', 'dept'])['salary'].mean().reset_index()
grouped

,city,dept,salary
0,LA,eng,100000.0
1,LA,sales,60000.0
2,NYC,eng,62500.0
3,SF,sales,90000.0


In [59]:
# Iterate over groups
for city, group_df in df.groupby('city'):
    print(f'=== {city} ===')
    print(group_df)
    print()

=== LA ===
   name  age   salary city   dept
e2  Bob   30  60000.0   LA  sales

=== NYC ===
     name  age   salary city dept
e1  Alice   25  50000.0  NYC  eng
e3  Carol   35  75000.0  NYC  eng

=== SF ===
    name  age   salary city   dept
e4  Dave   40  90000.0   SF  sales



### Quiz — Multi-Select

**Q: Which statements are correct? (more than one may be correct)**

1. ☐ `groupby()` can group by multiple columns
2. ☐ `.agg()` can apply multiple functions to multiple columns
3. ☐ `groupby()` returns a DataFrame directly
4. ☐ `.agg()` requires the data to be sorted first

---

**Answers:** 1 ✓ and 2 ✓

### Concept 1 — `groupby()` can group by multiple columns

Yes. Pass a LIST of column names. The result is grouped hierarchically (a MultiIndex).

```python
df.groupby(['city', 'dept'])['salary'].mean()
```

Each unique combination of (city, dept) becomes one group:

```
(NYC, eng)   →  group 1
(NYC, sales) →  group 2
(LA, eng)    →  group 3
(LA, sales)  →  group 4
(SF, sales)  →  group 5
```

The result has a **MultiIndex** — combining both grouping keys.

### Concept 2 — `.agg()` can apply multiple functions to multiple columns

Yes. Two flexible forms:

**Same functions on all columns:**
```python
df.groupby('city').agg(['mean', 'min', 'max'])
```
Applies all three functions to every numeric column.

**Different functions per column (dict):**
```python
df.groupby('city').agg({
    'salary': ['mean', 'max'],     # multiple functions on salary
    'age':    'mean'                # single function on age
})
```
This is the most flexible form — pick exactly which functions on which columns.

### Concept 3 — `groupby()` returns a DataFrame directly  ✗

FALSE. `groupby()` returns a **DataFrameGroupBy** object — a *lazy* grouping structure. It hasn't computed anything yet.

```python
g = df.groupby('city')
type(g)   # <class 'pandas.core.groupby.DataFrameGroupBy'>
```

You need to chain an aggregation (`.mean()`, `.agg()`, `.sum()`, etc.) to get an actual DataFrame or Series.

### Concept 4 — `.agg()` requires sorting  ✗

FALSE. No sorting needed — `groupby()` finds groups regardless of order. (You CAN pass `sort=False` to speed it up, but sorting is internal and optional.)

In [ ]:
# Demonstrate concept 1 — groupby with multiple columns
df.groupby(['city', 'dept'])['salary'].mean()

In [ ]:
# Demonstrate concept 2 — agg with multiple functions on multiple columns
df.groupby('city').agg({
    'salary': ['mean', 'max'],
    'age':    ['mean', 'min']
})

In [ ]:
# Demonstrate concept 3 — groupby() returns a GroupBy object, not a DataFrame
g = df.groupby('city')
print(type(g))
print('---')
print(type(g.mean(numeric_only=True)))   # NOW it's a DataFrame

## 5 — map, apply, applymap, transform

Four ways to apply a function. **Easy to confuse — here's the breakdown.**

| Method | Works on | Input → Output |
|--------|----------|----------------|
| `Series.map` | Single Series | Element-wise transformation |
| `Series.apply` | Single Series | Element-wise (more flexible than map) |
| `DataFrame.apply` | DataFrame | Whole row OR whole column |
| `DataFrame.applymap` | DataFrame | Element-wise (deprecated in newer pandas; use `df.map`) |
| `DataFrame.transform` | DataFrame / groups | Element-wise, RETURNS SAME SHAPE |

### 5a — Series.map

Element-wise on a single column. Takes a function OR a dict for value substitution.

In [80]:
df['age']

0    25
1    30
2    35
3    40
4    45
Name: age, dtype: int64

In [81]:
# Function — apply to each value
df['age'].map(lambda x: x * 2)

0    50
1    60
2    70
3    80
4    90
Name: age, dtype: int64

In [82]:
df["city"]

0    NYC
1     LA
2    NYC
3     SF
4     LA
Name: city, dtype: object

In [83]:
# Dict — substitute values
df['city'].map({'NYC': 'New York', 'LA': 'Los Angeles', 'SF': 'San Francisco'})

0         New York
1      Los Angeles
2         New York
3    San Francisco
4      Los Angeles
Name: city, dtype: object

### 5b — Series.apply

Like `map` for Series, but more flexible. Can return multiple values.

In [84]:
# Same as map
df['age'].apply(lambda x: x * 2)

0    50
1    60
2    70
3    80
4    90
Name: age, dtype: int64

In [ ]:
# More flexible — pass extra args
def tax(salary, rate):
    return salary * rate

df['salary'].apply(tax, rate=0.3)
# better actually is df['salary'] * 0.3

### 5c — DataFrame.apply

Applies a function to **rows** (axis=1) or **columns** (axis=0).

```
axis=0 (default)  → apply to each COLUMN (column-wise)
axis=1            → apply to each ROW (row-wise)
```

In [85]:
df[['age', 'salary']]

,age,salary
0,25,50000
1,30,60000
2,35,75000
3,40,90000
4,45,100000


In [88]:
# axis=0 — column-wise
df[['age', 'salary']].apply(lambda col: col.max() - col.min())
# returns one value PER COLUMN

age          20
salary    50000
dtype: int64

In [91]:
result = df[['age', 'salary']].apply(lambda col: col.max() - col.min()).reset_index()
result

,index,0
0,age,20
1,salary,50000


In [94]:
result = result.rename(columns={"index": "feature", 0: "range"})
result

,feature,range
0,age,20
1,salary,50000


In [ ]:
df[['age', 'salary']].apply(
    lambda col: pd.Series({
        "range": col.max() - col.min(),
        "mean": col.mean()
    })
)
# df[['age', 'salary']].agg([
#     ('range', lambda c: c.max() - c.min()),
#     ('mean',  'mean')
# ])

,age,salary
range,20.0,50000.0
mean,35.0,75000.0


1. df.agg('mean')
2. df.agg(['mean', 'max'])
3. df.agg({'col1': 'mean', 'col2': 'max'})
4. df.agg({'col': ['mean', 'max']})
5. df.agg(new_col=('col', 'func'))

In [87]:
# axis=1 — row-wise
df.apply(lambda row: f"{row['name']} ({row['age']})", axis=1)
# returns one value PER ROW

0    Alice (25)
1      Bob (30)
2    Carol (35)
3     Dave (40)
4      Eve (45)
dtype: object

### 5d — DataFrame.applymap (now DataFrame.map)

Element-wise on **every cell** of a DataFrame.

**Note:** `applymap` is deprecated in pandas ≥ 2.1 — use `DataFrame.map` instead.

In [ ]:
# Element-wise on each cell
nums = df[['age', 'salary']]
nums.map(lambda x: x * 2)    # use applymap on older pandas

### 5e — transform

Like `apply`, but the result **must have the same shape** as the input. Particularly useful with `groupby` — adds a column based on group-level statistics WITHOUT collapsing the data.

In [95]:
df['salary']

0     50000
1     60000
2     75000
3     90000
4    100000
Name: salary, dtype: int64

In [96]:
df['salary']/1000

0     50.0
1     60.0
2     75.0
3     90.0
4    100.0
Name: salary, dtype: float64

In [97]:
# Without groupby — element-wise with shape preservation
df['salary'].transform(lambda x: x / 1000)

0     50.0
1     60.0
2     75.0
3     90.0
4    100.0
Name: salary, dtype: float64

**Critical groupby use case** — add the GROUP MEAN as a column:

In [98]:
# Add average salary PER CITY as a new column
df['city_avg_salary'] = df.groupby('city')['salary'].transform('mean')
df

,name,age,salary,city,dept,city_avg_salary
0,Alice,25,50000,NYC,eng,62500.0
1,Bob,30,60000,LA,sales,80000.0
2,Carol,35,75000,NYC,eng,62500.0
3,Dave,40,90000,SF,sales,90000.0
4,Eve,45,100000,LA,eng,80000.0


Compare with the regular `groupby` result:

```
df.groupby('city')['salary'].mean()   → COLLAPSES to one row per city
df.groupby('city')['salary'].transform('mean')  → KEEPS original shape, broadcasts mean to each row
```

`transform` is what you want when you need to **merge group statistics back into the original DataFrame**.

## Quick Reference — Decision Guide

```
Want to transform a SINGLE COLUMN element-wise → series.map() or series.apply()

Want to apply a function ROW-BY-ROW             → df.apply(..., axis=1)

Want to apply a function COLUMN-BY-COLUMN       → df.apply(..., axis=0)

Want element-wise across the ENTIRE DataFrame   → df.map() (or applymap on older pandas)

Want group-level stats merged back              → df.groupby(...).transform('mean')

Want to aggregate per group (collapse rows)     → df.groupby(...).agg('mean')
```

## Summary

**Series & DataFrame** — 1D and 2D labelled data structures.

**Indexing:**
- `loc[label]` → label-based, slice endpoint included
- `iloc[pos]` → position-based, slice endpoint excluded
- Lists for fancy indexing, boolean masks for filtering

**groupby:**
- Split → Apply → Combine
- `.agg()` for collapsing; `.transform()` for keeping shape

**Function application:**
- `map` — Series, element-wise (or dict substitution)
- `apply` — Series element-wise; DataFrame row/column-wise
- `applymap` / `df.map` — DataFrame, element-wise
- `transform` — same shape output, perfect with groupby

These are the bread-and-butter operations of pandas — master them and 80% of data wrangling is fast and intuitive.